<a href="https://colab.research.google.com/github/khaliqtaimoor6-codes/Coronary-artery-stenosis-segmentation-/blob/main/00_setup_utils.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Setup & shared utilities
Run this **once** (and again only if you change a module). It
1. mounts Drive and creates the project folder tree,
2. writes the shared modules to `stenosis_research/src/` (every later notebook just does `import ...`),
3. finds your ARCADE folder and stores its path,
4. logs the environment,
5. runs a **smoke test** (metrics unit tests + a tiny train / crash / resume cycle on synthetic data).

**Use a GPU runtime** (Runtime → Change runtime type → T4 or better).

Folder tree created:
```
MyDrive/stenosis_research/
  src/  configs/  data/{processed,splits}/  runs/  results/  figures/  logs/
```
Each run lives in `runs/<nb>_<name>_s<seed>/` with `best.pt`, `last.pt`, `config.json`, `metrics.csv`, `final_metrics.json`, `probs_*.npz`.
`results/results.csv` gets one row per finished run; `results/test_access_log.csv` records every touch of a test split.

In [4]:
import os
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/stenosis_research'
ARCADE_ROOT_HINT = None      # e.g. '/content/drive/MyDrive/arcade'  (leave None to auto-search)
os.makedirs(ROOT + '/src', exist_ok=True)
print('project root:', ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project root: /content/drive/MyDrive/stenosis_research


In [5]:
!pip -q install segmentation-models-pytorch scikit-learn scipy pandas matplotlib

## Shared modules (written to Drive)
Edit them here, re-run the cell, and every notebook picks up the change. Don't edit the files on Drive by hand, or this notebook and Drive will disagree.

In [6]:
%%writefile /content/drive/MyDrive/stenosis_research/src/config.py
"""Central paths. Every notebook does: sys.path.insert(0, SRC); import config as C"""
import json, os
from pathlib import Path

DRIVE_ROOT = Path(os.environ.get("STENOSIS_ROOT", "/content/drive/MyDrive/stenosis_research"))
SRC        = DRIVE_ROOT / "src"
CONFIGS    = DRIVE_ROOT / "configs"
PROCESSED  = DRIVE_ROOT / "data" / "processed"   # cached npz arrays
SPLITS     = DRIVE_ROOT / "data" / "splits"      # frozen splits.json lives here
RUNS       = DRIVE_ROOT / "runs"
RESULTS    = DRIVE_ROOT / "results"
FIGS       = DRIVE_ROOT / "figures"
LOGS       = DRIVE_ROOT / "logs"
LOCAL      = Path("/content/local_cache")        # fast local disk (wiped on disconnect)

SPLITS_FILE = SPLITS / "splits.json"
RESULTS_CSV = RESULTS / "results.csv"
TEST_LOG    = RESULTS / "test_access_log.csv"

SIZES = (128, 256, 512)      # cached resolutions (512 = native ARCADE)
TASKS = ("stenosis", "syntax")

def make_dirs():
    for p in (SRC, CONFIGS, PROCESSED, SPLITS, RUNS, RESULTS, FIGS, LOGS, LOCAL):
        p.mkdir(parents=True, exist_ok=True)

def load_paths():
    f = CONFIGS / "paths.json"
    return json.load(open(f)) if f.exists() else {}

def save_paths(d):
    CONFIGS.mkdir(parents=True, exist_ok=True)
    json.dump(d, open(CONFIGS / "paths.json", "w"), indent=2)

def arcade_root():
    r = load_paths().get("arcade_root")
    if not r:
        raise RuntimeError("arcade_root not set. Run notebook 00 first.")
    return Path(r)


Overwriting /content/drive/MyDrive/stenosis_research/src/config.py


In [7]:
%%writefile /content/drive/MyDrive/stenosis_research/src/metrics.py
"""Numpy-only metrics. All metrics are computed from binary arrays (N,H,W)."""
import numpy as np
from scipy import ndimage

EPS = 1e-9

def _counts(pred, gt):
    pred = pred.astype(bool); gt = gt.astype(bool)
    tp = int(np.logical_and(pred, gt).sum())
    fp = int(np.logical_and(pred, ~gt).sum())
    fn = int(np.logical_and(~pred, gt).sum())
    return tp, fp, fn

def prf_from_counts(tp, fp, fn):
    p = tp / (tp + fp + EPS); r = tp / (tp + fn + EPS)
    f1 = 2 * p * r / (p + r + EPS)
    iou = tp / (tp + fp + fn + EPS)
    return dict(iou=iou, precision=p, recall=r, f1=f1)

def pooled_metrics(pred, gt):
    """Pixels pooled over the whole set (the protocol used by both reference papers:
    IoU = F1/(2-F1) holds exactly)."""
    return prf_from_counts(*_counts(pred, gt))

def per_image_metrics(pred, gt):
    """Mean/std of per-image Dice and IoU. Both-empty images count as 1.0."""
    pred = pred.astype(bool); gt = gt.astype(bool)
    tp = np.logical_and(pred, gt).sum((1, 2)).astype(np.float64)
    fp = np.logical_and(pred, ~gt).sum((1, 2)).astype(np.float64)
    fn = np.logical_and(~pred, gt).sum((1, 2)).astype(np.float64)
    denom_d = 2 * tp + fp + fn; denom_i = tp + fp + fn
    dice = np.where(denom_d == 0, 1.0, 2 * tp / np.maximum(denom_d, 1))
    iou = np.where(denom_i == 0, 1.0, tp / np.maximum(denom_i, 1))
    pos = gt.sum((1, 2)) > 0
    return dict(dice_mean=float(dice.mean()), dice_std=float(dice.std()),
                iou_mean=float(iou.mean()), iou_std=float(iou.std()),
                dice_mean_pos=float(dice[pos].mean()) if pos.any() else float("nan"),
                n_images=int(len(dice)), n_pos_images=int(pos.sum())), dice

SIZE_BINS = ((0, 100), (100, 300), (300, 1000), (1000, 10**9))   # GT lesion area in px at 512x512

def lesion_level(pred, gt, iou_thr=0.5):
    """Lesion-level detection: connected components, greedy IoU matching (>= iou_thr).
    Recall is also reported by GT lesion size (areas normalised to a 512x512 image)."""
    pred = pred.astype(bool); gt = gt.astype(bool)
    scale = (512.0 / pred.shape[-1]) ** 2
    TP = FP = FN = 0
    bin_tp = [0] * len(SIZE_BINS); bin_n = [0] * len(SIZE_BINS)
    st = np.ones((3, 3), int)
    for i in range(len(pred)):
        gl, ng = ndimage.label(gt[i], structure=st)
        pl, npd = ndimage.label(pred[i], structure=st)
        if ng == 0 and npd == 0:
            continue
        cnt = np.bincount((gl.astype(np.int64) * (npd + 1) + pl).ravel(),
                          minlength=(ng + 1) * (npd + 1)).reshape(ng + 1, npd + 1)
        ga = cnt.sum(1); pa = cnt.sum(0)
        cand = []
        for g in range(1, ng + 1):
            for p in range(1, npd + 1):
                o = cnt[g, p]
                if o > 0:
                    iou = o / (ga[g] + pa[p] - o)
                    if iou >= iou_thr:
                        cand.append((iou, g, p))
        cand.sort(reverse=True)
        mg, mp = set(), set()
        for _, g, p in cand:
            if g not in mg and p not in mp:
                mg.add(g); mp.add(p)
        TP += len(mg); FP += npd - len(mp); FN += ng - len(mg)
        for g in range(1, ng + 1):
            area = ga[g] * scale
            for b, (lo, hi) in enumerate(SIZE_BINS):
                if lo <= area < hi:
                    bin_n[b] += 1; bin_tp[b] += int(g in mg); break
    out = prf_from_counts(TP, FP, FN)
    out.update(tp=TP, fp=FP, fn=FN)
    out["recall_by_size"] = {f"{lo}-{hi if hi < 10**9 else 'inf'}px": (bin_tp[b] / bin_n[b] if bin_n[b] else float("nan"))
                             for b, (lo, hi) in enumerate(SIZE_BINS)}
    out["n_gt_by_size"] = {f"{lo}-{hi if hi < 10**9 else 'inf'}px": bin_n[b] for b, (lo, hi) in enumerate(SIZE_BINS)}
    return out

def sweep_threshold(probs, gt, thrs=None):
    """Pooled F1 at each threshold. Returns (best_thr, table)."""
    thrs = np.round(np.arange(0.10, 0.901, 0.05), 2) if thrs is None else thrs
    table = []
    for t in thrs:
        m = pooled_metrics(probs >= t, gt); m["thr"] = float(t); table.append(m)
    best = max(table, key=lambda m: m["f1"])
    return best["thr"], table

def evaluate_probs(probs, gt, thr=0.5, lesion=True):
    pred = probs >= thr
    pi, _ = per_image_metrics(pred, gt)
    out = dict(thr=float(thr), pooled=pooled_metrics(pred, gt), per_image=pi)
    if lesion:
        out["lesion"] = lesion_level(pred, gt)
    return out


Writing /content/drive/MyDrive/stenosis_research/src/metrics.py


In [8]:
%%writefile /content/drive/MyDrive/stenosis_research/src/arcade_io.py
"""ARCADE (COCO format) -> arrays. No torch needed."""
import json, hashlib
from pathlib import Path
import numpy as np, cv2

def _find_dir(root: Path, name: str):
    root = Path(root)
    for p in [root / name] + [q for q in root.rglob("*") if q.is_dir() and q.name.lower() == name.lower()]:
        if p.is_dir():
            return p
    return None

def find_arcade_root(hint=None, search_from="/content/drive/MyDrive", max_depth=4):
    """Return a directory that contains 'stenosis' and 'syntax' sub-directories."""
    def ok(p):
        p = Path(p)
        return p.is_dir() and (p / "stenosis").is_dir() and (p / "syntax").is_dir()
    cands = [hint] if hint else []
    cands += [Path(search_from) / "arcade", Path(search_from) / "ARCADE",
              Path(search_from) / "arcade" / "arcade"]
    for c in cands:
        if c and ok(c):
            return Path(c)
    base = Path(search_from); base_depth = len(base.parts)
    import os
    for dp, dn, _ in os.walk(base):
        if len(Path(dp).parts) - base_depth >= max_depth:
            dn[:] = []; continue
        if ok(dp):
            return Path(dp)
    raise FileNotFoundError("Could not find an ARCADE folder containing 'stenosis' and 'syntax'. "
                            "Set ARCADE_ROOT manually in notebook 00.")

def split_paths(root, task, split):
    """-> (images_dir, annotation_json) for e.g. task='stenosis', split='train'."""
    d = _find_dir(Path(root) / task, split)
    if d is None:
        raise FileNotFoundError(f"{root}/{task}/{split} not found")
    img_dir = d / "images"
    ann_dir = d / "annotations"
    js = sorted(ann_dir.glob("*.json"))
    if not img_dir.is_dir() or not js:
        raise FileNotFoundError(f"Expected {img_dir} and a .json in {ann_dir}")
    return img_dir, js[0]

def load_coco(json_path):
    coco = json.load(open(json_path))
    return coco["images"], coco["annotations"], coco.get("categories", [])

def rasterize(anns, h, w):
    """Union of all annotation polygons -> uint8 {0,1} mask. Returns (mask, n_skipped)."""
    m = np.zeros((h, w), np.uint8); skipped = 0
    for a in anns:
        seg = a.get("segmentation")
        if isinstance(seg, list) and len(seg) > 0:
            for poly in seg:
                pts = np.asarray(poly, dtype=np.float64).reshape(-1, 2)
                if len(pts) >= 3:
                    cv2.fillPoly(m, [np.round(pts).astype(np.int32)], 1)
                else:
                    skipped += 1
        else:
            skipped += 1      # RLE / empty segmentation: reported, not silently ignored
    return m, skipped

def build_split_arrays(root, task, split):
    """Read every image + build binary mask. Returns dict with images (N,H,W) uint8, masks, keys, stats."""
    img_dir, js = split_paths(root, task, split)
    images, anns, _ = load_coco(js)
    by_img = {}
    for a in anns:
        by_img.setdefault(a["image_id"], []).append(a)
    images = sorted(images, key=lambda d: d["id"])
    X, Y, keys, skipped, missing = [], [], [], 0, []
    for im in images:
        p = img_dir / im["file_name"]
        g = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if g is None:
            missing.append(im["file_name"]); continue
        if g.shape != (512, 512):
            g = cv2.resize(g, (512, 512), interpolation=cv2.INTER_LINEAR)
        m, s = rasterize(by_img.get(im["id"], []), im["height"], im["width"])
        if m.shape != (512, 512):
            m = cv2.resize(m, (512, 512), interpolation=cv2.INTER_NEAREST)
        X.append(g); Y.append(m); keys.append(f"{split}/{im['file_name']}"); skipped += s
    return dict(images=np.stack(X), masks=np.stack(Y), keys=np.array(keys),
                n_annotations=len(anns), n_skipped_annotations=skipped, missing=missing)

def resize_stack(arr, size, is_mask):
    if arr.shape[-1] == size:
        return arr.copy()
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR   # same as DCD-VMUNet's protocol
    return np.stack([cv2.resize(a, (size, size), interpolation=interp) for a in arr])

def ahash_bits(img, hs=16):
    s = cv2.resize(img, (hs, hs), interpolation=cv2.INTER_AREA).astype(np.float32)
    return (s > s.mean()).ravel()

def canonical_sha256(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()


Writing /content/drive/MyDrive/stenosis_research/src/arcade_io.py


In [9]:
%%writefile /content/drive/MyDrive/stenosis_research/src/utils.py
import os, sys, json, random, platform, subprocess, hashlib, time
from pathlib import Path
import numpy as np
import config as C

def seed_everything(seed: int, deterministic: bool = False):
    import torch
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic

def get_device():
    import torch
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def env_info():
    import torch
    info = dict(python=platform.python_version(), torch=torch.__version__,
                cuda=torch.version.cuda, platform=platform.platform(),
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
    try:
        import segmentation_models_pytorch as smp; info["smp"] = smp.__version__
    except Exception:
        pass
    return info

def load_splits(verify=True):
    """Load the frozen split file. Raises if the content hash does not match."""
    s = json.load(open(C.SPLITS_FILE))
    if verify:
        from arcade_io import canonical_sha256
        h = canonical_sha256({"tasks": s["tasks"], "folds": s["folds"]})
        if h != s["sha256"]:
            raise RuntimeError("splits.json has been modified! hash mismatch.")
    return s

def get_cache(task, size, local=True):
    """Return dict(images, masks, keys) for a task at a resolution. Copies npz to local disk once (fast)."""
    src = C.PROCESSED / f"{task}_{size}.npz"
    dst = C.LOCAL / f"{task}_{size}.npz"
    if local:
        C.LOCAL.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            import shutil; shutil.copy(src, dst)
        src = dst
    z = np.load(src, allow_pickle=False)
    return dict(images=z["images"], masks=z["masks"], keys=z["keys"])

def get_split_data(task, size, split, splits=None):
    """Arrays for one official split of a task, in the order frozen in splits.json."""
    splits = splits or load_splits()
    cache = get_cache(task, size)
    index = {k: i for i, k in enumerate(cache["keys"].tolist())}
    idx = [index[k] for k in splits["tasks"][task][split]]
    return cache["images"][idx], cache["masks"][idx]

def make_data_dict(size, task="stenosis", extra=("test",)):
    """{'train':(X,Y), 'val':(X,Y), 'extra': {'test':(X,Y)}} - the format run_experiment expects."""
    s = load_splits()
    d = {"train": get_split_data(task, size, "train", s), "val": get_split_data(task, size, "val", s), "extra": {}}
    for e in extra:
        d["extra"][e] = get_split_data(task, size, e, s)
    return d

def log_test_access(run, note=""):
    """Every evaluation on a test split is recorded here, so test-set use is auditable."""
    import csv
    C.RESULTS.mkdir(parents=True, exist_ok=True)
    new = not C.TEST_LOG.exists()
    with open(C.TEST_LOG, "a", newline="") as f:
        w = csv.writer(f)
        if new: w.writerow(["time", "run", "note"])
        w.writerow([time.strftime("%Y-%m-%d %H:%M:%S"), run, note])


Writing /content/drive/MyDrive/stenosis_research/src/utils.py


In [10]:
%%writefile /content/drive/MyDrive/stenosis_research/src/losses.py
import torch, torch.nn as nn, torch.nn.functional as F

def _dice_loss(prob, target, smooth=1.0):
    # pooled over the whole batch (same form as eq. 7 of Iman et al. / DCD-VMUNet)
    inter = (prob * target).sum()
    return 1 - (2 * inter + smooth) / (prob.sum() + target.sum() + smooth)

class BCEDice(nn.Module):
    """DCD-VMUNet protocol loss: BCE + Dice."""
    def __init__(self, smooth=1.0): super().__init__(); self.smooth = smooth
    def forward(self, logits, target):
        return F.binary_cross_entropy_with_logits(logits, target) + _dice_loss(torch.sigmoid(logits), target, self.smooth)

class WCEDice(nn.Module):
    """Iman et al.: alpha*WCE + (1-alpha)*Dice. pos_weight = stenosis class weight (15 in the paper)."""
    def __init__(self, pos_weight=15.0, alpha=0.5, smooth=1e-6):
        super().__init__(); self.register_buffer("pw", torch.tensor(float(pos_weight))); self.alpha = alpha; self.smooth = smooth
    def forward(self, logits, target):
        wce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=self.pw)
        return self.alpha * wce + (1 - self.alpha) * _dice_loss(torch.sigmoid(logits), target, self.smooth)

class FocalTversky(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1.0):
        super().__init__(); self.a, self.b, self.g, self.s = alpha, beta, gamma, smooth
    def forward(self, logits, target):
        p = torch.sigmoid(logits)
        tp = (p * target).sum(); fp = (p * (1 - target)).sum(); fn = ((1 - p) * target).sum()
        ti = (tp + self.s) / (tp + self.a * fn + self.b * fp + self.s)
        return (1 - ti) ** self.g

def build_loss(name, **kw):
    return {"bce_dice": BCEDice, "wce_dice": WCEDice, "focal_tversky": FocalTversky}[name](**kw)


Writing /content/drive/MyDrive/stenosis_research/src/losses.py


In [11]:
%%writefile /content/drive/MyDrive/stenosis_research/src/data.py
import random, math
import numpy as np, cv2, torch
from torch.utils.data import Dataset

AUG_PRESETS = {
    "none": None,
    # DCD-VMUNet protocol: hflip 0.5, vflip 0.5, rotation 0-360 deg with p=0.5
    "dcd":  dict(hflip=.5, vflip=.5, rot360=.5),
    # Iman et al. (Table 3): flips, rot90, shift +-5%, scale +-10%, rot +-15, brightness/contrast +-0.2
    "iman": dict(hflip=.5, vflip=.5, rot90=.5, shift=.05, scale=.10, rot=15, bright=.2, contrast=.2),
    # a stronger recipe for later Track-B experiments
    "strong": dict(hflip=.5, vflip=.5, rot90=.5, shift=.05, scale=.10, rot=15, bright=.2, contrast=.2),
}

def _affine(img, mask, angle, scale, tx, ty):
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale); M[:, 2] += (tx * w, ty * h)
    img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    mask = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return img, mask

def augment(img, mask, cfg):
    """img: float32 HxW in [0,1], mask: uint8 HxW. Image+mask always transformed together."""
    if random.random() < cfg.get("hflip", 0): img, mask = img[:, ::-1], mask[:, ::-1]
    if random.random() < cfg.get("vflip", 0): img, mask = img[::-1], mask[::-1]
    if random.random() < cfg.get("rot90", 0):
        k = random.randint(1, 3); img, mask = np.rot90(img, k), np.rot90(mask, k)
    img, mask = np.ascontiguousarray(img), np.ascontiguousarray(mask)
    if random.random() < cfg.get("rot360", 0):
        img, mask = _affine(img, mask, random.uniform(0, 360), 1.0, 0, 0)
    if any(k in cfg for k in ("shift", "scale", "rot")):
        ang = random.uniform(-cfg.get("rot", 0), cfg.get("rot", 0))
        sc = 1 + random.uniform(-cfg.get("scale", 0), cfg.get("scale", 0))
        tx = random.uniform(-cfg.get("shift", 0), cfg.get("shift", 0)); ty = random.uniform(-cfg.get("shift", 0), cfg.get("shift", 0))
        img, mask = _affine(img, mask, ang, sc, tx, ty)
    if "bright" in cfg:
        img = img + random.uniform(-cfg["bright"], cfg["bright"])
    if "contrast" in cfg:
        c = 1 + random.uniform(-cfg["contrast"], cfg["contrast"]); img = (img - img.mean()) * c + img.mean()
    return np.clip(img, 0, 1).astype(np.float32), mask

def worker_init_fn(_):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

class SegDataset(Dataset):
    """images uint8 (N,H,W), masks uint8 {0,1} (N,H,W)."""
    def __init__(self, images, masks, aug="none", in_ch=3):
        self.X, self.Y, self.in_ch = images, masks, in_ch
        self.cfg = AUG_PRESETS[aug] if isinstance(aug, str) else aug
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        img = self.X[i].astype(np.float32) / 255.0; m = self.Y[i].astype(np.uint8)
        if self.cfg: img, m = augment(img, m, self.cfg)
        x = torch.from_numpy(img)[None]
        if self.in_ch == 3: x = x.repeat(3, 1, 1)          # grayscale replicated, as in DCD-VMUNet
        return x, torch.from_numpy(np.ascontiguousarray(m)).float()[None]


Writing /content/drive/MyDrive/stenosis_research/src/data.py


In [12]:
%%writefile /content/drive/MyDrive/stenosis_research/src/models.py
import torch, torch.nn as nn, torch.nn.functional as F

def _block(i, o):
    # conv -> ReLU -> BN, twice (as described by Iman et al.)
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.ReLU(inplace=True), nn.BatchNorm2d(o),
                         nn.Conv2d(o, o, 3, padding=1), nn.ReLU(inplace=True), nn.BatchNorm2d(o))

class ImanUNet(nn.Module):
    """U-Net as described in Iman et al.: 4 encoder blocks (64,128,256,512), bottleneck 1024,
    dropout 0.3/0.3/0.4/0.5 after pooling, 4 decoder blocks. Output: 1 logit channel (the paper
    used a 2-channel softmax; equivalent for the binary case)."""
    def __init__(self, in_ch=3, classes=1, f=(64, 128, 256, 512), drops=(0.3, 0.3, 0.4, 0.5)):
        super().__init__()
        self.enc = nn.ModuleList(); c = in_ch
        for o in f: self.enc.append(_block(c, o)); c = o
        self.drops = nn.ModuleList([nn.Dropout2d(d) for d in drops])
        self.bott = _block(f[-1], f[-1] * 2)
        self.up = nn.ModuleList(); self.dec = nn.ModuleList(); c = f[-1] * 2
        for o in reversed(f):
            self.up.append(nn.ConvTranspose2d(c, o, 2, stride=2)); self.dec.append(_block(o * 2, o)); c = o
        self.head = nn.Conv2d(f[0], classes, 1)
    def forward(self, x):
        skips = []
        for e, d in zip(self.enc, self.drops):
            x = e(x); skips.append(x); x = d(F.max_pool2d(x, 2))
        x = self.bott(x)
        for up, dec, s in zip(self.up, self.dec, reversed(skips)):
            x = dec(torch.cat([up(x), s], 1))
        return self.head(x)

class Normalised(nn.Module):
    """Applies ImageNet mean/std inside the model so the data pipeline stays identical for all models."""
    def __init__(self, net, imagenet=False):
        super().__init__(); self.net = net; self.imagenet = imagenet
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        if self.imagenet: x = (x - self.mean) / self.std
        return self.net(x)

def build_model(cfg):
    arch = cfg["arch"]; in_ch = cfg.get("in_ch", 3); weights = cfg.get("weights")
    if arch == "iman_unet":
        return Normalised(ImanUNet(in_ch, 1), imagenet=False)
    import segmentation_models_pytorch as smp
    cls = {"unet": smp.Unet, "unetpp": smp.UnetPlusPlus, "deeplabv3plus": smp.DeepLabV3Plus,
           "fpn": smp.FPN, "manet": smp.MAnet}[arch]
    net = cls(encoder_name=cfg.get("encoder", "resnet34"), encoder_weights=weights, in_channels=in_ch, classes=1)
    return Normalised(net, imagenet=(weights == "imagenet" and in_ch == 3))


Writing /content/drive/MyDrive/stenosis_research/src/models.py


In [13]:
%%writefile /content/drive/MyDrive/stenosis_research/src/train.py
"""Generic, resumable trainer used by every notebook.
    res = run_experiment(cfg, data, nb="02")
data = {'train': (X,Y), 'val': (X,Y), 'extra': {'test': (X,Y), ...}}   (uint8 arrays, masks in {0,1})
"""
import os, json, time, random, platform
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
import config as C
from data import SegDataset, worker_init_fn
from models import build_model
from losses import build_loss
from metrics import evaluate_probs, sweep_threshold
from utils import seed_everything, get_device, env_info, log_test_access

DEFAULTS = dict(arch="unet", encoder="resnet34", weights=None, in_ch=3, size=256,
                loss="bce_dice", loss_kw={}, optimizer="adamw", lr=1e-4, wd=1e-2, min_lr=1e-5,
                sched="cosine", milestones=[], gamma=0.1, epochs=100, batch_size=16, eval_bs=32,
                aug="dcd", amp=True, num_workers=2, seed=42, save_every=5, patience=None,
                save_probs=True, tta=False)

def _atomic_save(obj, path):
    tmp = str(path) + ".tmp"; torch.save(obj, tmp); os.replace(tmp, path)

def _load(path):
    try: return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError: return torch.load(path, map_location="cpu")

def _loader(X, Y, cfg, train):
    ds = SegDataset(X, Y, cfg["aug"] if train else "none", cfg["in_ch"])
    g = torch.Generator(); g.manual_seed(cfg["seed"])
    nw = cfg["num_workers"]
    return DataLoader(ds, batch_size=cfg["batch_size"] if train else cfg["eval_bs"], shuffle=train,
                      drop_last=bool(train and len(ds) >= cfg["batch_size"]), num_workers=nw,
                      pin_memory=torch.cuda.is_available(), worker_init_fn=worker_init_fn, generator=g,
                      persistent_workers=nw > 0)

def _autocast(dev, enabled):
    return torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=enabled and dev.type == "cuda")

@torch.no_grad()
def predict_probs(model, X, cfg, dev, tta=None):
    tta = cfg["tta"] if tta is None else tta
    model.eval(); out = []
    for x, _ in DataLoader(SegDataset(X, np.zeros_like(X), "none", cfg["in_ch"]), batch_size=cfg["eval_bs"], shuffle=False):
        x = x.to(dev)
        def fwd(z):
            with _autocast(dev, cfg["amp"]): return torch.sigmoid(model(z).float())
        p = fwd(x)
        if tta: p = (p + fwd(x.flip(-1)).flip(-1) + fwd(x.flip(-2)).flip(-2)) / 3
        out.append(p[:, 0].cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def _make_opt(cfg, params):
    o = cfg["optimizer"]
    if o == "adamw":   return torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg["wd"])
    if o == "adam":    return torch.optim.Adam(params, lr=cfg["lr"], weight_decay=cfg["wd"])
    if o == "rmsprop": return torch.optim.RMSprop(params, lr=cfg["lr"], weight_decay=cfg["wd"])
    raise ValueError(o)

def _make_sched(cfg, opt):
    s = cfg["sched"]
    if s == "cosine":    return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=cfg["min_lr"])
    if s == "multistep": return torch.optim.lr_scheduler.MultiStepLR(opt, milestones=cfg["milestones"], gamma=cfg["gamma"])
    return None

def _flat_row(res):
    row = dict(run=res["run"], nb=res["nb"], name=res["name"], seed=res["seed"], size=res["cfg"]["size"],
               arch=res["cfg"]["arch"], encoder=res["cfg"]["encoder"], weights=res["cfg"]["weights"],
               loss=res["cfg"]["loss"], aug=res["cfg"]["aug"], epochs_run=res["epochs_run"],
               best_epoch=res["best_epoch"], thr=res["thr"], best_ckpt=res["best_ckpt"])
    for sname, d in res["sets"].items():
        row[f"{sname}_f1@0.5"] = d["thr0.5"]["pooled"]["f1"]
        for k in ("f1", "iou", "precision", "recall"): row[f"{sname}_{k}"] = d["tuned"]["pooled"][k]
        row[f"{sname}_dice_img"] = d["tuned"]["per_image"]["dice_mean"]
        row[f"{sname}_lesion_f1"] = d["tuned"]["lesion"]["f1"]
    return row

def append_results(row):
    C.RESULTS.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(C.RESULTS_CSV, dtype={'nb': str}) if C.RESULTS_CSV.exists() else pd.DataFrame()
    if len(df) and "run" in df.columns: df = df[df["run"] != row["run"]]
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(C.RESULTS_CSV, index=False)

def run_experiment(cfg, data, nb="xx", force=False, verbose=True):
    cfg = {**DEFAULTS, **cfg}
    run = f"{nb}_{cfg['name']}_s{cfg['seed']}"
    rd = C.RUNS / run; rd.mkdir(parents=True, exist_ok=True)
    final = rd / "final_metrics.json"
    if final.exists() and not force:
        if verbose: print(f"[{run}] already finished -> loading saved metrics")
        return json.load(open(final))
    json.dump(dict(cfg=cfg, env=env_info()), open(rd / "config.json", "w"), indent=2)

    seed_everything(cfg["seed"]); dev = get_device()
    model = build_model(cfg).to(dev)
    loss_fn = build_loss(cfg["loss"], **cfg["loss_kw"]).to(dev)
    opt = _make_opt(cfg, model.parameters()); sched = _make_sched(cfg, opt)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg["amp"] and dev.type == "cuda")
    (Xtr, Ytr), (Xva, Yva) = data["train"], data["val"]
    tl, vl = _loader(Xtr, Ytr, cfg, True), _loader(Xva, Yva, cfg, False)

    start, best, best_ep, bad = 0, -1.0, -1, 0
    last = rd / "last.pt"
    if last.exists():
        ck = _load(last)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        if sched and ck.get("sched"): sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        start, best, best_ep = ck["epoch"] + 1, ck["best"], ck["best_epoch"]
        torch.set_rng_state(ck["rng"]["torch"]); np.random.set_state(ck["rng"]["numpy"]); random.setstate(ck["rng"]["python"])
        if verbose: print(f"[{run}] resumed from epoch {start} (best F1 so far {best:.4f})")

    def save_last(ep):
        _atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict() if sched else None,
                          scaler=scaler.state_dict(), epoch=ep, best=best, best_epoch=best_ep, cfg=cfg,
                          rng=dict(torch=torch.get_rng_state(), numpy=np.random.get_state(), python=random.getstate())), last)

    log_f = rd / "metrics.csv"; ep = start - 1
    for ep in range(start, cfg["epochs"]):
        t0 = time.time(); model.train(); tr_loss = 0.0; n = 0
        for x, y in tl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with _autocast(dev, cfg["amp"]): logits = model(x)
            loss = loss_fn(logits.float(), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tr_loss += loss.item() * len(x); n += len(x)
        if sched: sched.step()
        # --- validation (pooled F1 at 0.5): model selection uses VAL only ---
        model.eval(); tp = fp = fn = 0.0; vloss = 0.0; vn = 0
        with torch.no_grad():
            for x, y in vl:
                x, y = x.to(dev), y.to(dev)
                with _autocast(dev, cfg["amp"]): lg = model(x)
                lg = lg.float(); vloss += loss_fn(lg, y).item() * len(x); vn += len(x)
                pr = (torch.sigmoid(lg) >= 0.5).float()
                tp += (pr * y).sum().item(); fp += (pr * (1 - y)).sum().item(); fn += ((1 - pr) * y).sum().item()
        f1 = 2 * tp / (2 * tp + fp + fn + 1e-9)
        row = dict(epoch=ep, train_loss=tr_loss / n, val_loss=vloss / vn, val_f1=f1, lr=opt.param_groups[0]["lr"], sec=time.time() - t0)
        pd.DataFrame([row]).to_csv(log_f, mode="a", header=not log_f.exists(), index=False)
        improved = f1 > best
        if improved:
            best, best_ep, bad = f1, ep, 0
            _atomic_save(dict(model=model.state_dict(), epoch=ep, val_f1=f1, cfg=cfg), rd / "best.pt")
        else:
            bad += 1
        if verbose: print(f"[{run}] ep {ep+1:3d}/{cfg['epochs']} loss {row['train_loss']:.4f} val_loss {row['val_loss']:.4f} "
                          f"val_f1 {f1:.4f}{' *' if improved else ''} ({row['sec']:.0f}s)")
        if (ep + 1) % cfg["save_every"] == 0 or ep + 1 == cfg["epochs"]: save_last(ep)
        if cfg["patience"] and bad >= cfg["patience"]:
            if verbose: print(f"[{run}] early stop");
            save_last(ep); break

    # ---------------- final evaluation with the BEST (val-selected) checkpoint ----------------
    model.load_state_dict(_load(rd / "best.pt")["model"]); model.to(dev)
    sets = {"val": (Xva, Yva), **data.get("extra", {})}
    probs = {k: predict_probs(model, X, cfg, dev) for k, (X, _) in sets.items()}
    thr, _ = sweep_threshold(probs["val"], Yva)                  # threshold chosen on VAL only
    out_sets = {}
    for k, (_, Y) in sets.items():
        out_sets[k] = {"thr0.5": evaluate_probs(probs[k], Y, 0.5), "tuned": evaluate_probs(probs[k], Y, thr)}
        if k.startswith("test"): log_test_access(run, "final eval of val-selected checkpoint")
        if cfg["save_probs"]: np.savez_compressed(rd / f"probs_{k}.npz", probs=probs[k].astype(np.float16))
    res = dict(run=run, nb=nb, name=cfg["name"], seed=cfg["seed"], cfg=cfg, epochs_run=ep + 1, best_epoch=best_ep,
               best_val_f1=best, thr=thr, best_ckpt=str(rd / "best.pt"), sets=out_sets)
    json.dump(res, open(final, "w"), indent=2, default=float)
    append_results(_flat_row(res))
    if verbose:
        for k, d in out_sets.items():
            print(f"[{run}] {k}: pooled F1@0.5={d['thr0.5']['pooled']['f1']:.4f}  F1@thr{thr:.2f}={d['tuned']['pooled']['f1']:.4f}  "
                  f"IoU={d['tuned']['pooled']['iou']:.4f}  lesionF1={d['tuned']['lesion']['f1']:.4f}")
    del model; torch.cuda.empty_cache()
    return res


Writing /content/drive/MyDrive/stenosis_research/src/train.py


## Configure paths and check the dataset

In [14]:
import sys, importlib, json
sys.path.insert(0, ROOT + '/src')
import config as C; importlib.reload(C)
C.make_dirs()
import arcade_io as A
arcade_root = A.find_arcade_root(ARCADE_ROOT_HINT)
C.save_paths({'arcade_root': str(arcade_root)})
print('ARCADE found at:', arcade_root)
for t in C.TASKS:
    for s in ('train', 'val', 'test'):
        img_dir, js = A.split_paths(arcade_root, t, s)
        print(f'  {t:9s} {s:5s} {len(list(img_dir.glob("*.png"))):5d} png   {js.name}')

ARCADE found at: /content/drive/MyDrive/arcade
  stenosis  train  1000 png   train.json
  stenosis  val     200 png   val.json
  stenosis  test    300 png   test.json
  syntax    train  1000 png   train.json
  syntax    val     200 png   val.json
  syntax    test    300 png   test.json


In [15]:
# Environment log (reproducibility)
import subprocess, torch
from utils import env_info
info = env_info(); print(info)
json.dump(info, open(C.LOGS / 'env_info.json', 'w'), indent=2)
open(C.LOGS / 'pip_freeze.txt', 'w').write(subprocess.run(['pip', 'freeze'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU! Switch the Colab runtime to GPU.'

{'python': '3.13.15', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'gpu': 'Tesla T4', 'smp': '0.5.0'}


## Smoke test 1 · metrics (pure numpy)

In [16]:
import numpy as np, metrics as M
gt = np.zeros((4, 64, 64), bool); gt[0, 10:20, 10:20] = 1; gt[1, 30:34, 30:50] = 1; gt[2, 5:9, 5:9] = 1; gt[2, 40:50, 40:50] = 1
pr = gt.copy(); pr[1] = False; pr[3, 0:5, 0:5] = True; pr[0, 10:20, 10:15] = False
p = M.pooled_metrics(pr, gt)
assert abs(p['iou'] - p['f1'] / (2 - p['f1'])) < 1e-6              # IoU = F1/(2-F1) for pooled pixels
ll = M.lesion_level(pr, gt)
assert (ll['tp'], ll['fp'], ll['fn']) == (3, 1, 1), ll              # 1 hit @IoU=.5, 1 miss, 1 false alarm, 1 exact
print('metrics OK', {k: round(v, 3) for k, v in p.items()})

metrics OK {'iou': 0.517, 'precision': 0.869, 'recall': 0.561, 'f1': 0.682}


## Smoke test 2 · trainer on synthetic data (train → "crash" → resume)
Uses a throwaway folder, so nothing from this test lands in your real `runs/` or `results.csv`.

In [17]:
import importlib, shutil
from pathlib import Path
import numpy as np
import train as T; importlib.reload(T)
C_runs, C_res, C_csv, C_tlog = C.RUNS, C.RESULTS, C.RESULTS_CSV, C.TEST_LOG
tmp = Path('/content/smoke'); shutil.rmtree(tmp, ignore_errors=True)
C.RUNS, C.RESULTS, C.RESULTS_CSV, C.TEST_LOG = tmp / 'runs', tmp / 'results', tmp / 'results/results.csv', tmp / 'results/test_log.csv'

rng = np.random.default_rng(0)
def fake(n, s=64):
    X = np.zeros((n, s, s), np.uint8); Y = np.zeros((n, s, s), np.uint8)
    for i in range(n):
        X[i] = rng.integers(60, 120, (s, s)); y, x = rng.integers(8, s - 16, 2); Y[i, y:y+8, x:x+4] = 1; X[i][Y[i] == 1] = 220
    return X, Y
data = dict(train=fake(24), val=fake(8), extra=dict(test=fake(8)))
base = dict(name='smoke', arch='unet', encoder='resnet18', weights=None, size=64, batch_size=8, eval_bs=8,
            num_workers=0, save_every=1, aug='dcd', lr=1e-3)
r1 = T.run_experiment({**base, 'epochs': 2}, data, nb='smoke')
(C.RUNS / 'smoke_smoke_s42' / 'final_metrics.json').unlink()           # pretend the session died after epoch 2
r2 = T.run_experiment({**base, 'epochs': 3}, data, nb='smoke')         # must say "resumed from epoch 2"
assert r2['epochs_run'] == 3
r3 = T.run_experiment({**base, 'name': 'smoke_iman', 'arch': 'iman_unet', 'loss': 'wce_dice', 'loss_kw': dict(pos_weight=15), 'aug': 'iman', 'epochs': 1}, data, nb='smoke')
assert (C.TEST_LOG).exists()                                           # test access was logged
print('\nTrainer smoke test passed. Files:', sorted(p.name for p in (C.RUNS / 'smoke_smoke_s42').iterdir()))
C.RUNS, C.RESULTS, C.RESULTS_CSV, C.TEST_LOG = C_runs, C_res, C_csv, C_tlog
shutil.rmtree(tmp, ignore_errors=True)

/content/drive/MyDrive/stenosis_research/src/train.py:99: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg["amp"] and dev.type == "cuda")


[smoke_smoke_s42] ep   1/2 loss 1.7836 val_loss 1.5861 val_f1 0.0301 * (6s)
[smoke_smoke_s42] ep   2/2 loss 1.5831 val_loss 1.6416 val_f1 0.0375 * (0s)
[smoke_smoke_s42] val: pooled F1@0.5=0.0375  F1@thr0.70=0.5108  IoU=0.3430  lesionF1=0.0952
[smoke_smoke_s42] test: pooled F1@0.5=0.0329  F1@thr0.70=0.0153  IoU=0.0077  lesionF1=0.0000


/content/drive/MyDrive/stenosis_research/src/train.py:99: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg["amp"] and dev.type == "cuda")


[smoke_smoke_s42] resumed from epoch 2 (best F1 so far 0.0375)
[smoke_smoke_s42] ep   3/3 loss 1.5361 val_loss 1.6470 val_f1 0.0636 * (0s)
[smoke_smoke_s42] val: pooled F1@0.5=0.0636  F1@thr0.55=0.3378  IoU=0.2032  lesionF1=0.0606
[smoke_smoke_s42] test: pooled F1@0.5=0.0480  F1@thr0.55=0.1315  IoU=0.0704  lesionF1=0.0000


/content/drive/MyDrive/stenosis_research/src/train.py:99: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg["amp"] and dev.type == "cuda")


[smoke_smoke_iman_s42] ep   1/1 loss 0.8350 val_loss 0.8420 val_f1 0.0000 * (5s)
[smoke_smoke_iman_s42] val: pooled F1@0.5=0.0000  F1@thr0.10=0.0155  IoU=0.0078  lesionF1=0.0000
[smoke_smoke_iman_s42] test: pooled F1@0.5=0.0000  F1@thr0.10=0.0155  IoU=0.0078  lesionF1=0.0000

Trainer smoke test passed. Files: ['best.pt', 'config.json', 'final_metrics.json', 'last.pt', 'metrics.csv', 'probs_test.npz', 'probs_val.npz']


## Done
Next: **01_data_eda**.

Conventions every notebook follows
* Split = `data/splits/splits.json` (frozen in notebook 01, hash-verified on every load).
* Model selection and threshold choice use **val only**; test results are produced by `run_experiment` once per run and logged.
* Pooled-pixel metrics are the headline (they match both reference papers); per-image and lesion-level metrics are reported alongside.